# Governed Function Calling with PII and Cost Guardrails

This example shows how to add deterministic governance guardrails around Gemini's function calling capabilities using [TealTiger](https://github.com/agentguard-ai/tealtiger) (Apache 2.0).

When Gemini agents make autonomous tool calls, you often need to:
- **Block PII** from leaking into tool arguments (SSNs, credit cards, emails)
- **Restrict which tools** the agent can call (allowlist)
- **Enforce cost budgets** per session
- **Produce audit trails** for compliance (SOC2, HIPAA, EU AI Act)

All governance runs deterministically (regex + policy rules) — no additional LLM call, under 2ms overhead per decision.

In [ ]:
%pip install -q google-genai tealtiger

In [ ]:
from google.colab import userdata
from google import genai

# Set your Gemini API key
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

## Define Tools for the Agent

We'll create a simple agent with two tools: `search_database` and `send_email`. In production, you'd want governance to ensure:
- No PII (like SSNs) gets passed to search queries
- Only authorized tools are callable
- Cost doesn't exceed budget

In [ ]:
# Define tools the agent can call
def search_database(query: str) -> str:
    """Search the internal database for information."""
    return f"Results for '{query}': [Employee records found]"


def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to}: {subject}"


def delete_records(table: str, condition: str) -> str:
    """Delete records from a database table. DANGEROUS."""
    return f"Deleted from {table} where {condition}"


tools = [search_database, send_email, delete_records]

## Set Up Governance Policies

TealTiger evaluates governance deterministically — no LLM in the governance path. Policies are defined declaratively:

In [ ]:
from tealtiger import TealEngine, GovernancePolicy, GovernanceMode

# Configure governance
engine = TealEngine(
    mode=GovernanceMode.ENFORCE,
    policies=[
        # Only allow search_database and send_email (block delete_records)
        GovernancePolicy.tool_allowlist(["search_database", "send_email"]),
        # Block any tool call containing PII in its arguments
        GovernancePolicy.pii_block(["ssn", "credit_card", "email"]),
        # Cap session cost at $0.10
        GovernancePolicy.cost_limit(max_per_session=0.10),
        # Detect API keys/tokens in arguments
        GovernancePolicy.secret_detection(),
    ],
)

print(f"Governance mode: {engine.mode.value}")
print(f"Policies: {[p.type for p in engine.policies]}")

## Governed Function Calling

We wrap Gemini's function calling with governance. Before each tool call executes, TealTiger evaluates all policies against the tool name and arguments.

In [ ]:
import json


def governed_tool_call(tool_name: str, tool_args: dict) -> str:
    """Execute a tool call with governance guardrails."""
    # Evaluate governance BEFORE execution
    decision = engine.evaluate(
        tool_name=tool_name,
        tool_args=json.dumps(tool_args),
    )

    print(f"  Governance: [{decision.action}] tool={tool_name} "
          f"reason={decision.reason_codes} "
          f"risk={decision.risk_score} "
          f"latency={decision.evaluation_time_ms:.2f}ms")

    if decision.action == "DENY":
        return f"[BLOCKED] Tool '{tool_name}' denied: {', '.join(decision.reason_codes)}"

    # Execute the tool
    tool_map = {
        "search_database": search_database,
        "send_email": send_email,
        "delete_records": delete_records,
    }
    result = tool_map[tool_name](**tool_args)
    return result

## Example 1: Clean Tool Call (Allowed)

A normal search query with no PII — passes all governance checks.

In [ ]:
print("--- Clean query (should ALLOW) ---")
result = governed_tool_call("search_database", {"query": "quarterly revenue 2025"})
print(f"  Result: {result}\n")

## Example 2: PII in Arguments (Blocked)

A search query containing an SSN — governance blocks it before execution.

In [ ]:
print("--- Query with SSN (should DENY) ---")
result = governed_tool_call("search_database", {"query": "lookup SSN 123-45-6789"})
print(f"  Result: {result}\n")

## Example 3: Unauthorized Tool (Blocked)

`delete_records` is not in the allowlist — denied regardless of arguments.

In [ ]:
print("--- Unauthorized tool (should DENY) ---")
result = governed_tool_call("delete_records", {"table": "users", "condition": "active=false"})
print(f"  Result: {result}\n")

## Example 4: Secret in Arguments (Blocked)

An API key accidentally passed as a tool argument — blocked before it leaks.

In [ ]:
print("--- Secret in args (should DENY) ---")
result = governed_tool_call("send_email", {
    "to": "ops@company.com",
    "subject": "Deploy config",
    "body": "Use this key: sk-abcdefghij1234567890abcdef"
})
print(f"  Result: {result}\n")

## Full Agent Loop with Gemini Interactions API

Now let's wire governance into a real Gemini agent conversation with function calling using the Interactions API.

In [ ]:
# Use Gemini with function calling + governance via Interactions API
interaction = client.interactions.create(
    model="gemini-2.0-flash",
    input="Search our database for Q3 2025 revenue numbers",
    config={
        "tools": [search_database, send_email, delete_records],
    },
)

# Process function calls through governance
if interaction.function_calls:
    for fc in interaction.function_calls:
        print(f"\nGemini wants to call: {fc.name}({dict(fc.args)})")
        result = governed_tool_call(fc.name, dict(fc.args))
        print(f"Result: {result}")
else:
    print(f"Gemini response: {interaction.text}")

## Inspect the Audit Trail

Every governance decision is recorded. This structured evidence is what compliance teams need for SOC2/HIPAA audits.

In [ ]:
print("=== Governance Audit Trail ===")
print(f"Total decisions: {len(engine.decisions)}")
print(f"Denials: {sum(1 for d in engine.decisions if d.action == 'DENY')}")
print(f"Session cost: ${engine.cumulative_cost:.4f}")
print()

for i, decision in enumerate(engine.decisions):
    print(f"  [{i+1}] {decision.action} | tool={decision.tool_name} | "
          f"reason={decision.reason_codes} | "
          f"risk={decision.risk_score} | "
          f"latency={decision.evaluation_time_ms:.2f}ms")

## Governance Modes

TealTiger supports three modes for safe rollout:

| Mode | Behavior |
|------|----------|
| **ENFORCE** | Evaluates policies, blocks violations |
| **MONITOR** | Evaluates policies, records decisions, allows all through (dry run) |
| **OBSERVE** | Skips evaluation, passes through with minimal audit |

Start with MONITOR in staging, switch to ENFORCE in production.

In [ ]:
# Switch to MONITOR mode — evaluate but don't block
engine.mode = GovernanceMode.MONITOR

print("--- MONITOR mode: PII query evaluated but allowed ---")
result = governed_tool_call("search_database", {"query": "lookup SSN 987-65-4321"})
print(f"  Result: {result}")
print(f"  (Decision recorded: {engine.decisions[-1].action} — would block in ENFORCE)")

## Summary

This example demonstrated how to:
1. Add PII detection to Gemini function call arguments
2. Restrict which tools the agent can call via allowlists
3. Enforce per-session cost budgets
4. Detect leaked secrets before they reach external systems
5. Produce structured audit trails for compliance

All governance is deterministic — no additional LLM call, under 2ms per decision.

**Resources:**
- [TealTiger GitHub](https://github.com/agentguard-ai/tealtiger)
- [TealTiger Docs](https://docs.tealtiger.ai)
- [PyPI](https://pypi.org/project/tealtiger/)
- [Gemini Function Calling Docs](https://ai.google.dev/gemini-api/docs/function-calling)